In [3]:
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langchain_groq import ChatGroq
from langchain_core.messages.utils import trim_messages,count_tokens_approximately
from dotenv import load_dotenv
import os
from langchain_core.messages import HumanMessage,BaseMessage
from typing import TypedDict,Annotated,List
from langgraph.checkpoint.memory import InMemorySaver


In [2]:
load_dotenv(dotenv_path=".env", override=True)
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
model = ChatGroq(
    model="llama-3.3-70b-versatile",  
    api_key=GROQ_API_KEY  
)

In [4]:
class chatstate(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]

In [5]:
MAX_TOKENS=150

In [ ]:
def chat(state:chatstate):
    messages=trim_messages(
        state['messages'],
        strategy='last',
        token_counter=count_tokens_approximately,
        max_tokens=MAX_TOKENS

    )
    print('Current Token Count ->', count_tokens_approximately(messages=messages))

    for message in messages:
        print(message.content)
    response=model.invoke(messages)
    return{'messages':[response]}

In [7]:
graph=StateGraph(chatstate)
graph.add_node('chat',chat)
graph.add_edge(START,'chat')
graph.add_edge('chat',END)

In [8]:
checkpointer=InMemorySaver()

In [9]:
chatbot=graph.compile(checkpointer=checkpointer)